In [ ]:
import torch
import pandas as pd
from torchmetrics import Accuracy
from algorithms.pso import PSO
from algorithms.ga import GA
from algorithms.cmaes import CMAES
from algorithms.de import DE
import matplotlib.pyplot as plt

from algorithms.minimize import minimize

In [ ]:
data = pd.read_csv("Data/winequality-red.csv", sep=";")

X_np = data[data.columns[:-1]].values
y_np = data[data.columns[-1]].values
y_np = y_np - 3

In [ ]:
X = torch.tensor(X_np, dtype=torch.float32).to("cuda")
y = torch.tensor(y_np, dtype=torch.long).to("cuda")

In [ ]:
loss = torch.nn.CrossEntropyLoss()

In [ ]:
def MLP(theta, x, n_classes, h_units):
        """tiny MLP"""
        
        _, n_features = x.shape
        
        w1, b1, w2, b2 = torch.split(theta, [n_features*h_units, h_units, h_units*n_classes, n_classes])
        h = torch.tanh(x @ w1.view(n_features,h_units) + b1)
        out = h @ w2.view(h_units, n_classes) + b2
        return out

def fitness_function(x, n_classes, h_units=32):

    def obj(pop):
        fitnesses = []
        for theta in pop:
            
            logits = MLP(theta, x, n_classes, h_units)
            
            fitness = loss(logits, y)
            fitnesses.append(fitness)
        return torch.stack(fitnesses)
    return obj

In [ ]:
n_classes  = len(y.unique())
n_features = X.shape[-1]
h_units    = 16

accuracy = Accuracy(task="multiclass", num_classes=n_classes).to("cuda")

In [ ]:
n_weights = n_features*h_units + h_units + h_units*n_classes + n_classes

In [ ]:
obj_function = fitness_function(X, n_classes, h_units=h_units)

In [ ]:
lower_bound = [-0.5]*n_weights
upper_bound = [0.5]*n_weights

# Adam

In [ ]:
n_epochs = int(1000)

lb = torch.tensor(lower_bound, device="cuda")
ub = torch.tensor(upper_bound, device="cuda")

mean = 0.5 * (lb + ub)          # centre of the box
std  = 0.5 * (ub - lb) / 3.0    # 3-σ rule  ⇒  99.7 % inside bounds
weights = mean + std * torch.randn(1, n_weights, device="cuda")
weights = torch.max(torch.min(weights, ub), lb)

weights = torch.nn.Parameter(weights.squeeze(0))
optimizer = torch.optim.Adam([weights], lr=0.001) 

In [ ]:
for epoch in range(n_epochs):
    optimizer.zero_grad(set_to_none=True)
    
    logits = MLP(weights, X, h_units=h_units, n_classes=n_classes)
    l = loss(logits, y)
    print(f"Epoch {epoch+1:4d} | Loss = {l.item():.4f}")
    l.backward()
    optimizer.step()

preds = MLP(weights, X, h_units=h_units, n_classes=n_classes)
accuracy(preds, y)

# PSO

In [ ]:
pop_size = 100
algorithm = PSO(obj_function, dim=n_weights, pop_size=pop_size, lower_bound=lower_bound, upper_bound=upper_bound, initialisation="gaussian")

_ = plt.hist(algorithm.pop[0].detach().cpu().numpy(), density=True)

preds = MLP(algorithm.best_x, X, h_units=h_units, n_classes=n_classes)
accuracy(preds, y)

In [ ]:
minimize(algorithm, max_evals=100000, verbose=True)

_ = plt.hist(algorithm.pop[0].detach().cpu().numpy(), density=True)

preds = MLP(algorithm.best_x, X, h_units=h_units, n_classes=n_classes)
accuracy(preds, y)

# GAs

In [ ]:
pop_size = 100
algorithm = GA(obj_function, dim=n_weights, pop_size=pop_size, lower_bound=lower_bound, upper_bound=upper_bound, initialisation="gaussian", mutation="gaussian", crossover="blend")

_ = plt.hist(algorithm.pop[0].detach().cpu().numpy(), density=True)

preds = MLP(algorithm.best_x, X, h_units=h_units, n_classes=n_classes)
accuracy(preds, y)

In [ ]:
minimize(algorithm, max_evals=10000, verbose=True)

_ = plt.hist(algorithm.pop[0].detach().cpu().numpy(), density=True)

preds = MLP(algorithm.best_x, X, h_units=h_units, n_classes=n_classes)
accuracy(preds, y)

# DE

In [ ]:
# pop_size = 100
# algorithm = DE(obj_function, dim=n_weights, pop_size=pop_size, lower_bound=lower_bound, upper_bound=upper_bound,, initialisation="gaussian")

# _ = plt.hist(algorithm.pop[0].detach().cpu().numpy(), density=True)

# preds = MLP(algorithm.best_x, X, h_units=h_units, n_classes=n_classes)
# accuracy(preds, y)

In [ ]:
# minimize(algorithm, max_evals=50000, verbose=True)

# _ = plt.hist(algorithm.pop[0].detach().cpu().numpy(), density=True)

# preds = MLP(algorithm.best_x, X, h_units=h_units, n_classes=n_classes)
# accuracy(preds, y)

# CMAES

In [ ]:
pop_size = 100
algorithm = CMAES(obj_function, dim=n_weights, pop_size=pop_size, lower_bound=lower_bound, upper_bound=upper_bound,)

_ = plt.hist(algorithm.pop[0].detach().cpu().numpy(), density=True)

preds = MLP(algorithm.best_x, X, h_units=h_units, n_classes=n_classes)
accuracy(preds, y)

In [ ]:
minimize(algorithm, max_evals=100000, verbose=True)

_ = plt.hist(algorithm.pop[0].detach().cpu().numpy(), density=True)

preds = MLP(algorithm.best_x, X, h_units=h_units, n_classes=n_classes)
accuracy(preds, y)